![Astrofisica Computacional](../new_logo.png)

# Computational Astrophysics – Datasets for crossmatching

## Dr. rer. nat. Jose Ivan Campos Rozo<sup>1,2</sup>

1. Astronomical Institute of the Czech Academy of Sciences\
   Department of Solar Physics\
   Ondřejov, Czec Republic

2. Observatorio Astronómico Nacional\
   Facultad de Ciencias\
   Universidad Nacional de Colombia

e-mail: jicamposr@unal.edu.co & rozo@asu.cas.cz)

---
Taken from previous lectures of this course.


### About this notebook

In this worksheet we introduce the datasets that we will work for the crossmatching.

---

This exercise of crossmatching two catalogues is based on the course *Data-Driven Astronomy* by the University of Sydney, available at

https://www.coursera.org/learn/data-driven-astronomy



### Introducing the Data

Before we can make the crossmatch of two catalogues we first have to import the raw data. 

Since every astronomy catalogue have its own unique format, we will need to load the data and study its structure individually. 

#### BSS Catalogue

The first catalogue that we need is the Australia Telescope 20-GHz **AT20G** Bright Source Sample (BSS) survey. This survey was carried out by the Australia Telescope Compact Array, from 2004 to 2007. In particular, the BSS is a complete flux-limited subsample of the AT20G Survey catalogue comprising 320 extragalactic radio sources.

The raw data is the file table2.dat that can be downloades from the VizieR archives,

http://cdsarc.u-strasbg.fr/viz-bin/Cat?J/MNRAS/384/775 


As we already shown, every catalogue in VizieR has a detailed README file that gives you the exact format of each table in the catalogue.

In [1]:
import numpy as np

The full BSS catalogue contains 320 objects. From the ReadMe file, we see that the contents in the table2.dat file include the features

| Column | Feature |
|---|---|
|1:| Object catalogue ID number (sometimes with an asterisk indicating a note)|
|2-4:| Right ascension in HMS notation|
|5-7:| Declination in DMS notation|
|8-:| Other information|


The information in the columns 8-  includes flux densities, spectral intensities, redshift, etc.

For crossmatching, we only need the coordinates. Therefore, we will load just the columns 2-7 with the `usecols` argument in the `numpy.loadtxt` function. 

Note: We will not load the ID column because this number is always the same as the row number.


In [2]:
bbs_raw = np.loadtxt('J_MNRAS_384_775/table2.dat', 
                     usecols=range(1, 7))
bbs_raw.shape

(320, 6)

In [3]:
bbs_raw[0:3]

array([[  0.  ,   4.  ,  35.65, -47.  ,  36.  ,  19.1 ],
       [  0.  ,  10.  ,  35.92, -30.  ,  27.  ,  48.3 ],
       [  0.  ,  11.  ,   1.27, -26.  ,  12.  ,  33.1 ]])

Since the Right Ascension is  given in hours-minutes-seconds (HMS) notation and the Declination is given in degrees-minutes-seconds (DMS) notation, we will use the functions defined before in this course to trasform this information to degrees.

In [4]:
def HMStoDeg(H,M,S):
  return 15*(H + M/60 + S/(60*60))

def DMStoDeg(D,M,S):
  if D>=0:
    return D + M/60 + S/(60*60)
  else:
    return -1*(-D + M/60 + S/(60*60))


Now, we will define an array with the position of the sources in degrees.

In [5]:
bss_cat = np.zeros([320,3])

for i in range(320):
  bss_cat[i,0]=i+1
  ascension = HMStoDeg(bbs_raw[i,0], bbs_raw[i,1], bbs_raw[i,2])
  declination = DMStoDeg(bbs_raw[i,3], bbs_raw[i,4], bbs_raw[i,5])
  bss_cat[i,1], bss_cat[i,2] = ascension, declination

bss_cat[0:5]

array([[  1.        ,   1.14854167, -47.60530556],
       [  2.        ,   2.64966667, -30.46341667],
       [  3.        ,   2.75529167, -26.20919444],
       [  4.        ,   3.24954167, -39.90733333],
       [  5.        ,   6.45491667, -26.03686111]])

---

#### SuperCOSMOS Catalogue

The SuperCOSMOS all-sky catalogue is a catalogue of galaxies generated from several visible light surveys available at 

http://ssa.roe.ac.uk/allSky

The complete description of the catalogue can be found in the paper

https://arxiv.org/abs/1607.01189

The raw data corresponds to the file 

http://ssa.roe.ac.uk/cats/SCOS_XSC_mCl1_B21.5_R20_noStepWedges.csv.gz

This dataset includes more than 241 millons of objects and its size is more than 8 Gb. 
 
Since this catalogue is so large, we will use a reduced version given in the course *Data-Driven Astronomy* by the University of Sydney, available at

https://www.coursera.org/learn/data-driven-astronomy

The corresponding file is available at

https://github.com/ashcat2005/data-driven-astronomy/blob/master/week2/data3/super.csv

and includes only 500 objects.

Although this a .csv file and can be read directly with `pandas`, we will use `numpy.loadtxt` to load the data as an array. 

The catalogue data includes in the first row the column labels. It is clear that the are

| Column | Feature |
|---|---|
|1:| Right ascension in decimal degrees|
|2:| Declination in decimal degrees|
|3:| Other data, including magnitude and apparent shape|


Since we only need the coordinates, we will load just the first two columns with the `usecols` argument in the `numpy.loadtxt` function. 

In [6]:
super_raw = np.loadtxt('super.csv', delimiter=',', 
                       skiprows=1, usecols=[0, 1])
super_raw.shape

(500, 2)

In [7]:
super_raw[0:3]

array([[ 6.03176000e-02, -8.56561327e+01],
       [ 1.14850820e+00, -4.76054131e+01],
       [ 1.27943310e+00, -1.54590140e+00]])

Since the coordinates in this dataset are already in degrees, we need no conversion. Now, we will define an array with the loaded data, including an index column,

In [8]:
super_cat = np.zeros([500,3])

for i in range(500):
  super_cat[i,0]=int(i+1)
  super_cat[i,1], super_cat[i,2] = super_raw[i,0], super_raw[i,1]

super_cat[0:5]

array([[ 1.00000000e+00,  6.03176000e-02, -8.56561327e+01],
       [ 2.00000000e+00,  1.14850820e+00, -4.76054131e+01],
       [ 3.00000000e+00,  1.27943310e+00, -1.54590140e+00],
       [ 4.00000000e+00,  2.64883140e+00, -3.04631581e+01],
       [ 5.00000000e+00,  2.75505950e+00, -2.62091826e+01]])


---
### Angular Distance between Two Objects in the Catalogues

The angular distance between between one object in the first catalogue with coordinates $(\alpha_1 , \delta_1)$ and another object in the second catalogue with coordinates $(\alpha_2, \delta_2)$, can be calculated using the Haversine formula,

\begin{equation}
d = 2 \arcsin \sqrt{\sin^2 \frac{\left| \delta_1 - \delta_2 \right| }{2} + \cos \delta_1 \cos \delta_2 \sin^2 \frac{\left| \alpha_1 - \alpha_2\right|}{2}}
\end{equation}

https://en.wikipedia.org/wiki/Haversine_formula

We implmented this formula as a function before,

In [9]:
def ang_dist(RA1, dec1, RA2, dec2):
  '''
  The arguments RA and dec are given in decimal degrees
  and the function returns the angular distance in
  decimal degrees
  '''
  # Transform the arguments from decimal degrees to radians
  RA1 = np.radians(RA1)
  dec1 = np.radians(dec1)
  RA2 = np.radians(RA2)
  dec2 = np.radians(dec2)

  a = np.sin(np.abs(dec1 - dec2)/2)**2
  b = np.cos(dec1)*np.cos(dec2)*np.sin(np.abs(RA1 - RA2)/2)**2
  d = 2*np.arcsin(np.sqrt(a + b))
  return np.degrees(d)

For example, the distance between the first object in the BSS catalogue and the first object in the SuperCOSMOS catalogue is

In [10]:
ang_dist(bss_cat[0,1], bss_cat[0,2], super_cat[0,1], super_cat[0,2])

np.float64(38.05168335509249)

---
## Crossmatching 

The crossmatching process consists in finding a correspondence between the objects in two catalogues.

The process will be as follows:

1. Select an object from the BSS catalogue

2. Go through all the objects in the SuperCOSMOS catalogue, calculating the angular distance to find the closest one to the BSS object.

3. Record the match of the closest objects (* see note below).

4. Repeat steps 1-3 for all the objects in the BSS catalogue.

Note: If the closest object is not within a given maximum distance, it is unlikely that the two objects are actually counterparts, and it may be more likely that they are just nearby objects.

This maximum distance is choosen depending on the uncertainty of the measured object positions in each catalogue.

### Exercises

1. Write a crossmatch function that crossmatches two catalogues within a maximum distance. It should return a list of matches and non-matches for the first catalogue against the second.

2. The list of matches contains tuples of the first and second catalogue object IDs and their distance. 
The list of non-matches contains the unmatched object IDs from the first catalogue only. Both lists should be ordered by the first catalogue's IDs.

3. Include a function to calculate the time taken (in seconds) to run the crossmatcher.

4. Using the BSS and SuperCOSMOS catalogues as input arguments, compute the crossmatch function using a maximum distance of 10 arcseconds.

5. Compute again the crosmatching function using the BSS and SuperCOSMOS catalogues as input arguments with a maximum distance of 5 arcseconds.

In [11]:
def crossmatch(cat1,cat2, max_dist=1.):
  '''
  Crossmatch two catalogues cat1 and cat2 calculating the angular 
  distance between all the entries and choosing the minimum within 
  a maximum distance of max_dist
  '''
  # array to store all the distances
  distances = np.zeros(len(cat2))
  # list to store the match results
  match_results = []
  # list to store the no-match results
  nomatch_results = []

  # Main Loop
  for i in range(len(cat1)):
    for j in range(len(cat2)):
      distances[j] = ang_dist(cat1[i,1], cat1[i,2], 
                              cat2[j,1], cat2[j,2])
    # Compare with the minimum distance allowed
    if distances.min() < max_dist:
      # Append a match
      match_results.append((i + 1, int(np.where(distances==distances.min())[0][0] + 1), float(distances.min())))
    else:
      # Append a no-match
      nomatch_results.append(i + 1)

  return match_results, nomatch_results


In [12]:
%%time
matches1, no_matches1 = crossmatch(bss_cat, super_cat)


CPU times: total: 641 ms
Wall time: 635 ms


In [13]:
matches1

[(1, 2, 0.00010988610938710059),
 (2, 4, 0.0007649845967242494),
 (3, 5, 0.00020863352870707666),
 (4, 6, 0.00012867299967083928),
 (7, 11, 7.666235090011008e-05),
 (8, 14, 0.00020833046101750643),
 (9, 15, 0.0001774015026088326),
 (10, 17, 0.00016079604439502975),
 (12, 18, 0.0003493944606752222),
 (13, 22, 0.00012170541597764247),
 (14, 24, 0.008314435904852577),
 (15, 25, 0.00021977416884437018),
 (16, 26, 0.0001328286890139291),
 (17, 27, 3.181530427842653e-05),
 (18, 29, 0.00019843774684364136),
 (19, 30, 0.00011585178516956528),
 (20, 31, 0.0002627136922326124),
 (21, 32, 0.00039932330328371096),
 (22, 33, 7.36535577225557e-05),
 (23, 34, 0.00022280948960772594),
 (24, 35, 0.00015481960583649628),
 (25, 37, 0.0003688060288937162),
 (26, 39, 0.0003750312712778308),
 (27, 41, 0.0001183449200434592),
 (28, 42, 0.00012217104062974945),
 (30, 43, 0.0005573032265457314),
 (31, 44, 0.00018231015875520086),
 (32, 46, 0.00023219797492993673),
 (33, 47, 3.915629971730293e-05),
 (34, 48, 0.

In [14]:
no_matches1

[5,
 6,
 11,
 29,
 58,
 82,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 168,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 186,
 187,
 188,
 189,
 190,
 191,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 205,
 206,
 208,
 209,
 210,
 211,
 212,
 213,
 214,
 215,
 217,
 218,
 220,
 221,
 222,
 223,
 224,
 227,
 230,
 231,
 232,
 233,
 234,
 235,
 236,
 237,
 238,
 239,
 242,
 245,
 247,
 248,
 250,
 251,
 252,
 253,
 254,
 255,
 256,
 259,
 262,
 263,
 264,
 265,
 266,
 267,
 268,
 270,
 271,
 272,
 273,
 274,
 275,
 276,
 277,
 278,
 279,
 281,
 282,
 284,
 285,
 286,
 287,
 288,
 289,
 290,
 291,
 292,
 293,
 294,
 295,
 296,
 297,
 298,
 299,
 300,
 301,
 302,
 303,
 304,
 305,
 306,
 307,
 308,
 309,
 310,
 311,
 312,
 314,
 315,
 316,
 317,
 318,
 319,
 320]

In [15]:
np.shape(matches1)

(180, 3)

In [16]:
np.shape(no_matches1)

(140,)

In [17]:
# Mejora 1: Convertir una sola vez los ángulos en grados a radianes
def crossmatch_radians(cat1, cat2, max_dist=1.):
    # Convertir todo a radianes una sola vez
    cat1_rad = np.radians(cat1[:, 1:3])
    cat2_rad = np.radians(cat2[:, 1:3])
    max_dist_rad = np.radians(max_dist)

    matches, no_matches = [], []
    for i, (ra1, dec1) in enumerate(cat1_rad):
        distances = haversine_vec(ra1, dec1, cat2_rad)
        min_idx = np.argmin(distances)
        if distances[min_idx] < max_dist_rad:
            matches.append( (int(cat1[i, 0]) , int(cat2[min_idx, 0]) , float(np.degrees(distances[min_idx]))) )
        else:
            no_matches.append(int(cat1[i, 0]))
    
    return matches, no_matches

# Mejora 2: Usar operaciones vectorizadas de numpy para el cálculo de distancias
def haversine_vec(ra1, dec1, cat2_rad):
    # Permite calcular la distancia angular entre un objeto del primer catálogo (ra1, dec1) y todos los
    # objetos del segundo catálogo de una vez
    ra2  = cat2_rad[:, 0]
    dec2 = cat2_rad[:, 1]

    # Haversine vectorizada, sin bucles explícitos:
    a = np.sin(0.5*np.abs(dec2 - dec1))**2
    b = np.cos(dec1) * np.cos(dec2) * np.sin(0.5*np.abs(ra2 - ra1))**2

    return 2 * np.arcsin(np.sqrt(a + b))


In [18]:
%%time
# Con las mejoras 1 y 2 se obtiene:
matches2, no_matches2 = crossmatch_radians(bss_cat, super_cat)

CPU times: total: 15.6 ms
Wall time: 16.7 ms


In [19]:
matches2

[(1, 2, 0.00010988610938710059),
 (2, 4, 0.0007649845967242494),
 (3, 5, 0.00020863352870707666),
 (4, 6, 0.00012867299967083928),
 (7, 11, 7.666235090011008e-05),
 (8, 14, 0.00020833046101750643),
 (9, 15, 0.0001774015026088326),
 (10, 17, 0.00016079604439502975),
 (12, 18, 0.0003493944606752222),
 (13, 22, 0.00012170541597764247),
 (14, 24, 0.008314435904852577),
 (15, 25, 0.00021977416884437018),
 (16, 26, 0.0001328286890139291),
 (17, 27, 3.181530427842653e-05),
 (18, 29, 0.00019843774684364136),
 (19, 30, 0.00011585178516956528),
 (20, 31, 0.0002627136922326124),
 (21, 32, 0.00039932330328371096),
 (22, 33, 7.36535577225557e-05),
 (23, 34, 0.00022280948960772594),
 (24, 35, 0.00015481960583649628),
 (25, 37, 0.0003688060288937162),
 (26, 39, 0.0003750312712778308),
 (27, 41, 0.0001183449200434592),
 (28, 42, 0.00012217104062974945),
 (30, 43, 0.0005573032265457314),
 (31, 44, 0.00018231015875520086),
 (32, 46, 0.00023219797492993673),
 (33, 47, 3.915629971730293e-05),
 (34, 48, 0.

In [20]:
no_matches2

[5,
 6,
 11,
 29,
 58,
 82,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 168,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 186,
 187,
 188,
 189,
 190,
 191,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 205,
 206,
 208,
 209,
 210,
 211,
 212,
 213,
 214,
 215,
 217,
 218,
 220,
 221,
 222,
 223,
 224,
 227,
 230,
 231,
 232,
 233,
 234,
 235,
 236,
 237,
 238,
 239,
 242,
 245,
 247,
 248,
 250,
 251,
 252,
 253,
 254,
 255,
 256,
 259,
 262,
 263,
 264,
 265,
 266,
 267,
 268,
 270,
 271,
 272,
 273,
 274,
 275,
 276,
 277,
 278,
 279,
 281,
 282,
 284,
 285,
 286,
 287,
 288,
 289,
 290,
 291,
 292,
 293,
 294,
 295,
 296,
 297,
 298,
 299,
 300,
 301,
 302,
 303,
 304,
 305,
 306,
 307,
 308,
 309,
 310,
 311,
 312,
 314,
 315,
 316,
 317,
 318,
 319,
 320]

In [21]:
np.shape(matches2)

(180, 3)

In [22]:
np.shape(no_matches2)

(140,)

In [23]:
# Mejora 3: Evitar el cálculo para los objetos del segundo catálogo que claramente estén fuera del rango deseado
# Para esto es necesario ordenar el catálogo 2 según su declinación

# Iniciamos la búsqueda dentro de cat2 para el objeto con dec2 > dec1 - r 
# Nos detenemos cuando dec2 < dec1 + r
# Siendo r el radio máximo del objeto con dec1.

def crossmatch_radians_sorted(cat1, cat2, max_dist=1.):
    # Convertir todo a radianes una sola vez
    cat1_rad = np.radians(cat1[:, 1:3])
    cat2_rad = np.radians(cat2[:, 1:3])
    max_dist_rad = np.radians(max_dist)

    # Implementar la mejora 3:
    sort = cat2[:, 2].argsort()
    cat2_sorted = cat2_rad[sort]
    idx_sorted = cat2[sort, 0]
    dec2_sorted = cat2_sorted[:, 1]

    matches, no_matches = [], []
    for i, (ra1, dec1) in enumerate(cat1_rad):
        start = np.searchsorted(dec2_sorted, dec1 - max_dist_rad, side='left' )
        end   = np.searchsorted(dec2_sorted, dec1 + max_dist_rad, side='right')
        subset = cat2_sorted[start:end]

        if start == end:
            no_matches.append(int(cat1[i, 0]))
            continue

        distances = haversine_vec(ra1, dec1, subset)
        min_idx   = np.argmin(distances) 
        if distances[min_idx] < max_dist_rad:
            matches.append( (int(cat1[i, 0]) , int(idx_sorted[start + min_idx]) , float(np.degrees(distances[min_idx]))) )
        else:
            no_matches.append(int(cat1[i, 0]))
    
    return matches, no_matches

In [24]:
%%time
# Con las mejoras 1, 2 y 3 se obtiene:
matches3, no_matches3 = crossmatch_radians_sorted(bss_cat, super_cat)

CPU times: total: 15.6 ms
Wall time: 10 ms


In [25]:
matches3

[(1, 2, 0.00010988610938710059),
 (2, 4, 0.0007649845967242494),
 (3, 5, 0.00020863352870707666),
 (4, 6, 0.00012867299967083928),
 (7, 11, 7.666235090011008e-05),
 (8, 14, 0.00020833046101750643),
 (9, 15, 0.0001774015026088326),
 (10, 17, 0.00016079604439502975),
 (12, 18, 0.0003493944606752222),
 (13, 22, 0.00012170541597764247),
 (14, 24, 0.008314435904852577),
 (15, 25, 0.00021977416884437018),
 (16, 26, 0.0001328286890139291),
 (17, 27, 3.181530427842653e-05),
 (18, 29, 0.00019843774684364136),
 (19, 30, 0.00011585178516956528),
 (20, 31, 0.0002627136922326124),
 (21, 32, 0.00039932330328371096),
 (22, 33, 7.36535577225557e-05),
 (23, 34, 0.00022280948960772594),
 (24, 35, 0.00015481960583649628),
 (25, 37, 0.0003688060288937162),
 (26, 39, 0.0003750312712778308),
 (27, 41, 0.0001183449200434592),
 (28, 42, 0.00012217104062974945),
 (30, 43, 0.0005573032265457314),
 (31, 44, 0.00018231015875520086),
 (32, 46, 0.00023219797492993673),
 (33, 47, 3.915629971730293e-05),
 (34, 48, 0.

In [26]:
no_matches3

[5,
 6,
 11,
 29,
 58,
 82,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 168,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 186,
 187,
 188,
 189,
 190,
 191,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 205,
 206,
 208,
 209,
 210,
 211,
 212,
 213,
 214,
 215,
 217,
 218,
 220,
 221,
 222,
 223,
 224,
 227,
 230,
 231,
 232,
 233,
 234,
 235,
 236,
 237,
 238,
 239,
 242,
 245,
 247,
 248,
 250,
 251,
 252,
 253,
 254,
 255,
 256,
 259,
 262,
 263,
 264,
 265,
 266,
 267,
 268,
 270,
 271,
 272,
 273,
 274,
 275,
 276,
 277,
 278,
 279,
 281,
 282,
 284,
 285,
 286,
 287,
 288,
 289,
 290,
 291,
 292,
 293,
 294,
 295,
 296,
 297,
 298,
 299,
 300,
 301,
 302,
 303,
 304,
 305,
 306,
 307,
 308,
 309,
 310,
 311,
 312,
 314,
 315,
 316,
 317,
 318,
 319,
 320]

In [27]:
np.shape(matches3)

(180, 3)

In [28]:
np.shape(no_matches3)

(140,)

In [29]:
print(matches1 == matches2, matches1 == matches3, matches2 == matches3)

True True True


Para el **ejercicio 3**, la función del tiempo:

In [30]:
import time

def tiempo_crossmatch(dmax):
    t0 = time.perf_counter()
    match, no_match = crossmatch_radians_sorted(bss_cat, super_cat, max_dist=dmax)
    return match, no_match, time.perf_counter() - t0

Para el **ejercicio 4** tomamos ```max_dist = 10''```.

In [31]:
match10, no_match10, t10 = tiempo_crossmatch(dmax=10/3600) 

print(f'Con max_dist = 10\'\' ({10/3600:.4f}°) se obtienen {len(match10)} aciertos y {len(no_match10)} fallos, con un tiempo de ejecución de t = {t10:.4f} s.')


Con max_dist = 10'' (0.0028°) se obtienen 131 aciertos y 189 fallos, con un tiempo de ejecución de t = 0.0073 s.


Para el **ejercicio 5** tomamos ```max_dist = 5''```.

In [32]:
match5, no_match5, t5 = tiempo_crossmatch(dmax=5/3600)

print(f'Con max_dist = 5\'\' ({5/3600:.4f}°) se obtienen {len(match5)} aciertos y {len(no_match5)} fallos, con un tiempo de ejecución de t = {t5:.4f} s.')


Con max_dist = 5'' (0.0014°) se obtienen 123 aciertos y 197 fallos, con un tiempo de ejecución de t = 0.0065 s.
